In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob

# find the folder wherever it lives (My Drive, shortcut, or Shared drives)
matches = (glob.glob('/content/drive/MyDrive/pokemon_wordle') +
           glob.glob('/content/drive/Shareddrives/*/pokemon_wordle') +
           glob.glob('/content/drive/MyDrive/**/pokemon_wordle', recursive=True))
print("Found:", matches)

PROJ = matches[0]
os.chdir(PROJ)
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)
os.makedirs(f'{PROJ}/cache', exist_ok=True)
print(PROJ, os.listdir(PROJ))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found: ['/content/drive/MyDrive/pokemon_wordle', '/content/drive/MyDrive/pokemon_wordle']
/content/drive/MyDrive/pokemon_wordle ['data', 'results', 'notebooks', 'slides', 'report', 'entropy_solver.py', 'minimax_solver.py', 'benchmark.py', 'app.py', 'README.txt', 'loader.py', 'engine.py', '__pycache__', 'basic_solvers.py', 'cache']


In [ ]:
%%writefile /content/drive/MyDrive/pokemon_wordle/entropy_solver.py
"""Entropy solver: filter candidates, then guess the Pokémon that splits them most evenly."""
import os, pickle, hashlib, math
import numpy as np

# ---- ADJUST THESE TWO IMPORTS IF YOUR TEAMMATES NAMED THINGS DIFFERENTLY ----
from engine import feedback
from loader import load_pokemon
# -----------------------------------------------------------------------------

CACHE_DIR = '/content/drive/MyDrive/pokemon_wordle/cache'


def _key(p):
    """Unique identifier of a Pokémon record (dict, pandas row, or object)."""
    if isinstance(p, dict):
        return p.get('name', p.get('Name', str(p)))
    for attr in ('name', 'Name'):
        if hasattr(p, attr):
            return getattr(p, attr)
    return str(p)


class EntropySolver:
    name = "Entropy"

    def __init__(self, pokemon=None, guess_pool="all", use_cache=True):
        """
        guess_pool = "all"        -> may guess any not-yet-guessed Pokémon (best splits)
        guess_pool = "candidates" -> may only guess Pokémon still possible
        """
        pokemon = load_pokemon() if pokemon is None else pokemon
        if hasattr(pokemon, 'to_dict'):              # DataFrame -> list of dicts
            pokemon = pokemon.to_dict('records')
        self.pokemon = list(pokemon)
        self.n = len(self.pokemon)
        self.guess_pool = guess_pool
        self.index = {_key(p): i for i, p in enumerate(self.pokemon)}
        self._first = None                           # first guess computed once
        self._build_matrix(use_cache)

    # ---------- precomputed clue matrix: M[g, a] = clue-pattern id of guess g vs answer a ----------
    def _build_matrix(self, use_cache):
        sig = hashlib.md5('|'.join(str(_key(p)) for p in self.pokemon).encode()).hexdigest()[:10]
        path = f'{CACHE_DIR}/clue_matrix_{self.n}_{sig}.pkl'
        if use_cache and os.path.exists(path):
            with open(path, 'rb') as f:
                self.M, self.patterns = pickle.load(f)
            return
        codes, M = {}, np.zeros((self.n, self.n), dtype=np.int32)
        for i, g in enumerate(self.pokemon):
            for j, a in enumerate(self.pokemon):
                fb = tuple(feedback(g, a))
                M[i, j] = codes.setdefault(fb, len(codes))
        self.M, self.patterns = M, codes
        if use_cache:
            os.makedirs(CACHE_DIR, exist_ok=True)
            with open(path, 'wb') as f:
                pickle.dump((M, codes), f)

    # ---------- helpers ----------
    def _idx(self, p):
        if isinstance(p, (int, np.integer)):
            return int(p)
        return self.index[p if isinstance(p, str) else _key(p)]

    def filter(self, cand, guess, pattern):
        """Keep only candidates that would have produced this exact feedback."""
        return cand[self.M[guess, cand] == pattern]

    def entropy(self, g, cand):
        counts = np.bincount(self.M[g, cand])
        counts = counts[counts > 0]
        p = counts / counts.sum()
        return float(-(p * np.log2(p)).sum())

    # ---------- the strategy ----------
    def choose(self, cand, guessed):
        """cand: np.array of candidate indices. guessed: set of indices already guessed."""
        if len(cand) == 1:
            return int(cand[0])
        if len(cand) == len(self.pokemon) and self._first is not None:
            return self._first                       # cached first guess
        cand_set = set(cand.tolist())
        pool = range(self.n) if (self.guess_pool == "all" and len(cand) > 2) else cand.tolist()
        best, best_score = None, -1.0
        for g in pool:
            if g in guessed:                         # never repeat a guess
                continue
            score = self.entropy(g, cand) + (1e-6 if g in cand_set else 0.0)  # tie -> prefer a possible answer
            if score > best_score:
                best, best_score = g, score
        if len(cand) == self.n:
            self._first = best
        return best

    # ---------- play a full game ----------
    def play(self, answer, max_guesses=50):
        """Returns (number_of_guesses, list_of_guessed_names)."""
        a = self._idx(answer)
        cand, guessed, history = np.arange(self.n), set(), []
        while len(history) < max_guesses:
            g = self.choose(cand, guessed)
            guessed.add(g)
            history.append(g)
            if g == a:
                return len(history), [_key(self.pokemon[i]) for i in history]
            cand = self.filter(cand, g, self.M[g, a])
        return max_guesses, [_key(self.pokemon[i]) for i in history]

    # ---------- interface for Member 5's app (works with Pokémon objects/names) ----------
    def next_guess(self, candidates, guessed=()):
        cand = np.array([self._idx(c) for c in candidates])
        gs = {self._idx(g) for g in guessed}
        return self.pokemon[self.choose(cand, gs)]

Overwriting /content/drive/MyDrive/pokemon_wordle/entropy_solver.py


In [ ]:
import importlib, time, numpy as np
import engine, loader, entropy_solver
for m in (engine, loader, entropy_solver): importlib.reload(m)
from entropy_solver import EntropySolver, _key

t = time.time()
S = EntropySolver()
print("Pokémon loaded:", S.n, "| matrix build/load time: %.1fs" % (time.time()-t))

# 1) one full game, should be well under a second
t = time.time()
n, hist = S.play(S.pokemon[0]['name'] if isinstance(S.pokemon[0], dict) else S.pokemon[0])
print("Game: %d guesses in %.3fs -> %s" % (n, time.time()-t, hist))

# 2) no repeated guesses, every game finishes
for a in range(S.n):
    n, h = S.play(a)
    assert len(h) == len(set(h)), f"repeated guess for answer {a}"
    assert h[-1] == _key(S.pokemon[a])
print("OK: no repeats, all games finish")

# 3) matrix filtering must match Member 2's filter_candidates (edit the call to match their signature)
try:
    from basic_solvers import filter_candidates
    g, a = 5, 17
    fb = engine.feedback(S.pokemon[g], S.pokemon[a])
    mine = set(S.filter(np.arange(S.n), g, S.M[g, a]).tolist())
    theirs = {S.index[_key(p)] for p in filter_candidates(S.pokemon, S.pokemon[g], fb)}
    print("Matches filter_candidates:", mine == theirs)
except Exception as e:
    print("Adjust the filter_candidates call to Member 2's signature:", e)

Pokémon loaded: 386 | matrix build/load time: 0.9s
Game: 3 guesses in 0.012s -> ['Croconaw', 'Oddish', 'Bulbasaur']
OK: no repeats, all games finish
Adjust the filter_candidates call to Member 2's signature: filter_candidates() takes 2 positional arguments but 3 were given


In [ ]:
import random, time, statistics as st
random.seed(0)

def filter_baseline(S, a):
    """Filter solver: random guess among consistent candidates."""
    cand, guessed, k = np.arange(S.n), set(), 0
    while True:
        g = int(random.choice(cand.tolist())); k += 1
        if g == a: return k
        cand = S.filter(cand, g, S.M[g, a])

t = time.time()
ent = [S.play(a)[0] for a in range(S.n)]
t_ent = time.time() - t
fil = [filter_baseline(S, a) for a in range(S.n)]

print("Entropy: avg %.2f | median %s | worst %d | %.2fs for all %d games" % (st.mean(ent), st.median(ent), max(ent), t_ent, S.n))
print("Filter : avg %.2f | median %s | worst %d" % (st.mean(fil), st.median(fil), max(fil)))
print("Entropy beats Filter:", st.mean(ent) < st.mean(fil))

# optional: candidates-only variant for your report
S2 = EntropySolver(guess_pool="candidates")
c = [S2.play(a)[0] for a in range(S2.n)]
print("Entropy (candidates only): avg %.2f" % st.mean(c))

Entropy: avg 3.07 | median 3.0 | worst 5 | 1.78s for all 386 games
Filter : avg 3.76 | median 4.0 | worst 8
Entropy beats Filter: True
Entropy (candidates only): avg 3.07
